<a href="https://colab.research.google.com/github/adarsh-crafts/BSDA5002-Mathematical-Foundations-of-Generative-AI/blob/main/04%20-%20Conditional%20GAN%20(CGAN)%20Implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Setup

In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import os
import matplotlib.pyplot as plt
import torchvision

In [ ]:
device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else 'cpu'
print(f"Using device: {device}")

Using device: mps


In [ ]:
output_dir = 'results/Conditional-GAN/'
os.makedirs(f'{output_dir}/generated_images', exist_ok=True)
os.makedirs(f'{output_dir}/saved_models', exist_ok=True)

## Config

In [ ]:
z_dim = 100
img_dim = 28
channels = 1
num_classes = 10

epochs = 10
batch_size = 128
learning_rate = 2e-4

steps_g = 3
steps_d = 1

## Load Data

In [ ]:
transformation = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, ), (0.5, ))
])

train_dataset = datasets.MNIST(root='data', train=True, download=False, transform=transformation)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

## Model Setup

- input to G would be z_dim + num_classes cos we input both and same for D.

In [ ]:
class Generator(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.label_emb = nn.Embedding(num_classes, num_classes)
        self.hidden = nn.Sequential(
            nn.ConvTranspose2d(input_dim + num_classes, out_channels = 256, kernel_size=7, stride=1, padding=0),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(True),

            nn.ConvTranspose2d(128, 1, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, x, labels):
        label = self.label_emb(labels)[:, :, None, None]   # (batch, num_classes) -> (batch, num_classes, 1, 1): to match latent vector (z) dimensions
        x = torch.cat([x, label], dim=1)
        return self.hidden(x)


class Discriminator(nn.Module):
    def __init__(self, img_dim, num_classes):
        super().__init__()
        self.img_dim = img_dim
        self.label_emb = nn.Embedding(num_classes, img_dim * img_dim)
        self.hidden = nn.Sequential(
            nn.Conv2d(2, 128, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, True),

            nn.Conv2d(256, 1, kernel_size=7, stride=1, padding=0),
            nn.Sigmoid()
        )

    def forward(self, x, labels):
        label = self.label_emb(labels).reshape(-1, 1, self.img_dim, self.img_dim)    # (batch, img_dim*img_dim) -> (batch, 1, img_dim, img_dim): to match image dimensions
        x = torch.cat([x, label], dim=1)
        return self.hidden(x)

In [ ]:
generator = Generator(z_dim, num_classes).to(device)
discriminator = Discriminator(img_dim, num_classes).to(device)

## Model Training

In [ ]:
optimizer_g = torch.optim.Adam(generator.parameters(), lr=learning_rate, betas=(0.5, 0.999))
optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=learning_rate, betas=(0.5, 0.999))

criterion = nn.BCELoss()

In [ ]:
for epoch in range(epochs):
    epoch_d_loss = 0.0
    epoch_g_loss = 0.0

    for batch_idx, (X, y) in enumerate(train_dataloader):
        X, y = X.to(device), y.to(device)
        batch_size = X.size(0)

        ## Train Discriminator
        for _ in range(steps_d):
            real_labels = torch.ones(batch_size, 1, 1, 1).to(device)
            fake_labels = torch.zeros(batch_size, 1, 1, 1).to(device)

            # loss fn term 1
            d_outputs_real = discriminator(X, y)
            d_loss_real = criterion(d_outputs_real, real_labels)

            # loss fn term 2
            z = torch.randn(batch_size, z_dim, 1, 1).to(device)
            g_outputs = generator(z, y)
            d_outputs_fake = discriminator(g_outputs.detach(), y)
            d_loss_fake = criterion(d_outputs_fake, fake_labels)

            # Total loss
            d_loss = d_loss_real + d_loss_fake
            optimizer_d.zero_grad()
            d_loss.backward()
            optimizer_d.step()


        ## Train Generator
        for _ in range(steps_g):
            z = torch.randn(batch_size, z_dim, 1, 1).to(device)
            g_outputs = generator(z, y)
            d_outputs = discriminator(g_outputs, y)
            g_loss = criterion(d_outputs, real_labels)

            optimizer_g.zero_grad()
            g_loss.backward()
            optimizer_g.step()

        epoch_d_loss += d_loss.item()
        epoch_g_loss += g_loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Discriminator Loss: {epoch_d_loss/len(train_dataloader):.4f}, Generator Loss: {epoch_g_loss/len(train_dataloader):.4f}")
    if (epoch+1) % 10 == 0 or epoch == 0:
        # Generate one sample for each digit (0-9)
        z = torch.randn(num_classes, z_dim, 1, 1).to(device)
        labels = torch.arange(num_classes, dtype=torch.long).to(device)

        with torch.no_grad():
            generated_images = generator(z, labels)
            generated_images = (generated_images + 1) / 2  # Rescale to [0, 1]

        grid = torchvision.utils.make_grid(generated_images, nrow=10, padding=2)
        plt.figure(figsize=(12, 2))
        plt.imshow(grid.permute(1, 2, 0).cpu().numpy(), cmap='gray')
        plt.axis('off')
        plt.savefig(f'{output_dir}/generated_images/epoch_{epoch+1}.png',
                    bbox_inches='tight', pad_inches=0)
        plt.close()


Epoch [1/10], Discriminator Loss: 1.3943, Generator Loss: 0.7240
Epoch [2/10], Discriminator Loss: 1.4003, Generator Loss: 0.7084
Epoch [3/10], Discriminator Loss: 1.4028, Generator Loss: 0.7033
Epoch [4/10], Discriminator Loss: 1.4038, Generator Loss: 0.7022
Epoch [5/10], Discriminator Loss: 1.4026, Generator Loss: 0.7002
Epoch [6/10], Discriminator Loss: 1.4032, Generator Loss: 0.6998
Epoch [7/10], Discriminator Loss: 1.4014, Generator Loss: 0.6993
Epoch [8/10], Discriminator Loss: 1.4002, Generator Loss: 0.6980
Epoch [9/10], Discriminator Loss: 1.4002, Generator Loss: 0.6990
Epoch [10/10], Discriminator Loss: 1.3990, Generator Loss: 0.6976


## Save Model

In [ ]:
torch.save(generator.state_dict(), f'{output_dir}/saved_models/generator.pth')